# Reusable Template: Multiple Variable Linear Regression for Economics Projects

A general-purpose, from-scratch multiple linear regression pipeline you can reuse for **any** economics dataset: consumption, wages, GDP growth, demand, house prices, inflation, etc.

**How to reuse this notebook:**
1. Edit only **Section 1 (Configuration)** — paste in your own data, feature names, and target name.
2. Run all cells top to bottom.
3. Everything downstream (scaling, gradient descent, evaluation, plots, interpretation) is generic and works unchanged.

# Outline
- [1 Configuration — EDIT THIS](#toc_1)
- [2 Setup](#toc_2)
- [3 Exploratory Look at the Data](#toc_3)
- [4 Feature Scaling](#toc_4)
- [5 Train / Test Split](#toc_5)
- [6 Model: Prediction, Cost, Gradient](#toc_6)
- [7 Run Gradient Descent](#toc_7)
- [8 Evaluate on Train & Test](#toc_8)
- [9 Coefficient Interpretation Table](#toc_9)
- [10 Predict on New Data](#toc_10)
- [11 Diagnostics: Residual Plots](#toc_11)


<a name="toc_1"></a>
# 1 Configuration — EDIT THIS SECTION FOR YOUR PROJECT

Replace the example data (household consumption) with your own. Requirements:
- `X_raw`: shape (m, n) — one row per observation, one column per explanatory variable
- `y_raw`: shape (m,) — the outcome you want to predict
- `feature_names`: list of `n` strings, matching the columns of `X_raw`
- `target_name`: string, name of the outcome variable


In [ ]:
import numpy as np

# ------------------------------------------------------------------
# EXAMPLE DATA (replace with your own economics dataset)
# Household consumption function:
#   features = [Monthly Income ($), Household Size, Head's Education (yrs), Head's Age (yrs)]
#   target   = Monthly Consumption Expenditure ($)
# ------------------------------------------------------------------
X_raw = np.array([
    [4360, 5, 16, 45],
    [2950, 3, 12, 40],
    [1560, 2,  9, 35],
    [3800, 4, 14, 50],
    [2100, 2, 10, 30],
    [5200, 6, 18, 55],
])
y_raw = np.array([3210, 2100, 1250, 2870, 1600, 3950])

feature_names = ['Monthly Income ($)', 'Household Size', "Head's Education (yrs)", "Head's Age (yrs)"]
target_name = 'Monthly Consumption Expenditure ($)'

# Gradient descent hyperparameters (tune these for your dataset)
ALPHA = 0.05          # learning rate (use a much larger value than the raw-scale lab since we scale features below)
NUM_ITERS = 2000       # number of gradient descent iterations
TEST_FRACTION = 0.2   # fraction of data held out for testing (use 0.0 if your dataset is very small)
RANDOM_SEED = 42

assert X_raw.shape[0] == y_raw.shape[0], "X_raw and y_raw must have the same number of rows"
assert X_raw.shape[1] == len(feature_names), "feature_names length must match number of columns in X_raw"
print(f"Loaded {X_raw.shape[0]} observations, {X_raw.shape[1]} features: {feature_names}")
print(f"Target: {target_name}")

<a name="toc_2"></a>
# 2 Setup


In [ ]:
import copy, math
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
rng = np.random.default_rng(RANDOM_SEED)

<a name="toc_3"></a>
# 3 Exploratory Look at the Data

Always look at ranges/scales before fitting — this tells you whether scaling is essential (it almost always is for economic data).


In [ ]:
m, n = X_raw.shape
print(f"m (observations) = {m}, n (features) = {n}\n")
print(f"{'Feature':30s}{'min':>10s}{'max':>10s}{'mean':>10s}{'std':>10s}")
for name, col in zip(feature_names, X_raw.T):
    print(f"{name:30s}{col.min():10.2f}{col.max():10.2f}{col.mean():10.2f}{col.std():10.2f}")
print(f"\n{target_name}: min={y_raw.min():.2f}, max={y_raw.max():.2f}, mean={y_raw.mean():.2f}")

fig, axes = plt.subplots(1, n, figsize=(4*n, 3.5), constrained_layout=True)
if n == 1:
    axes = [axes]
for ax, name, col in zip(axes, feature_names, X_raw.T):
    ax.scatter(col, y_raw)
    ax.set_xlabel(name)
    ax.set_ylabel(target_name if ax is axes[0] else '')
    ax.set_title(f'{name} vs target')
plt.show()

<a name="toc_4"></a>
# 4 Feature Scaling (z-score normalization)

Economic variables (income, prices, rates, counts) live on very different scales. Z-score normalization puts every feature on a comparable footing so gradient descent converges quickly and reliably.

$$ x_j^{norm} = \frac{x_j - \mu_j}{\sigma_j} $$

**Important:** compute `mu`/`sigma` only from the *training* data, then apply the same transform to test data and to any new prediction inputs — never let information from the test set leak into scaling statistics.


In [ ]:
def zscore_normalize(X, mu=None, sigma=None):
    '''Normalize columns of X to zero mean, unit std. If mu/sigma given, reuse them (for test/new data).'''
    if mu is None:
        mu = np.mean(X, axis=0)
    if sigma is None:
        sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

<a name="toc_5"></a>
# 5 Train / Test Split

Fit only on the training split; evaluate honestly on the held-out test split.


In [ ]:
def train_test_split(X, y, test_fraction, rng):
    m = X.shape[0]
    idx = rng.permutation(m)
    n_test = max(1, int(round(m * test_fraction))) if test_fraction > 0 else 0
    test_idx, train_idx = idx[:n_test], idx[n_test:]
    return X[train_idx], y[train_idx], X[test_idx], y[test_idx]

X_train_raw, y_train, X_test_raw, y_test = train_test_split(X_raw, y_raw, TEST_FRACTION, rng)
print(f"Train size: {X_train_raw.shape[0]}, Test size: {X_test_raw.shape[0]}")

# Scale using TRAIN statistics only
X_train, mu, sigma = zscore_normalize(X_train_raw)
if X_test_raw.shape[0] > 0:
    X_test, _, _ = zscore_normalize(X_test_raw, mu, sigma)
else:
    X_test = X_test_raw

<a name="toc_6"></a>
# 6 Model: Prediction, Cost, Gradient

These are generic — they work for any $n$ without modification.


In [ ]:
def predict(X, w, b):
    """Vectorized prediction for one or many examples. X: (n,) or (m,n)."""
    return X @ w + b

def compute_cost(X, y, w, b):
    """Mean-squared-error cost (halved), fully vectorized."""
    m = X.shape[0]
    errors = predict(X, w, b) - y
    return np.sum(errors ** 2) / (2 * m)

def compute_gradient(X, y, w, b):
    """Gradient of the cost w.r.t. w and b, fully vectorized."""
    m = X.shape[0]
    errors = predict(X, w, b) - y
    dj_dw = (X.T @ errors) / m
    dj_db = np.sum(errors) / m
    return dj_db, dj_dw

def gradient_descent(X, y, w_in, b_in, alpha, num_iters, verbose=True):
    """Batch gradient descent - returns final w, b, and cost history."""
    J_history = []
    w = copy.deepcopy(w_in)
    b = b_in
    print_every = max(1, math.ceil(num_iters / 10))

    for i in range(num_iters):
        dj_db, dj_dw = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        J_history.append(compute_cost(X, y, w, b))
        if verbose and i % print_every == 0:
            print(f"Iteration {i:5d}: Cost {J_history[-1]:12.4f}")

    return w, b, J_history

<a name="toc_7"></a>
# 7 Run Gradient Descent


In [ ]:
n_features = X_train.shape[1]
initial_w = np.zeros(n_features)
initial_b = 0.0

w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b, ALPHA, NUM_ITERS)

print(f"\nFinal b: {b_final:.4f}")
print("Final w:", w_final)

fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist)
tail_start = min(100, max(0, len(J_hist)//10))
ax2.plot(range(tail_start, len(J_hist)), J_hist[tail_start:])
ax1.set_title("Cost vs. iteration"); ax2.set_title("Cost vs. iteration (tail)")
ax1.set_xlabel("iteration"); ax2.set_xlabel("iteration")
ax1.set_ylabel("Cost"); ax2.set_ylabel("Cost")
plt.show()

**Diagnosing convergence:** the cost curve should fall smoothly and flatten out. If it oscillates or increases, `ALPHA` is too large — reduce it (try dividing by 3). If it's still falling steeply at the end, either increase `NUM_ITERS` or increase `ALPHA` slightly.


<a name="toc_8"></a>
# 8 Evaluate on Train & Test


In [ ]:
def evaluate(y_true, y_pred):
    err = y_pred - y_true
    mae = np.mean(np.abs(err))
    mse = np.mean(err ** 2)
    rmse = np.sqrt(mse)
    ss_res = np.sum(err ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

y_train_pred = predict(X_train, w_final, b_final)
train_metrics = evaluate(y_train, y_train_pred)
print("Train metrics:", {k: round(v, 3) for k, v in train_metrics.items()})

if X_test.shape[0] > 0:
    y_test_pred = predict(X_test, w_final, b_final)
    test_metrics = evaluate(y_test, y_test_pred)
    print("Test metrics: ", {k: round(v, 3) for k, v in test_metrics.items()})
else:
    print("No test split configured (TEST_FRACTION = 0).")

<a name="toc_9"></a>
# 9 Coefficient Interpretation Table

Because `w_final` is estimated on **scaled** features, its magnitude reflects "effect per 1-standard-deviation change," which is actually convenient for comparing relative importance across features measured in different units. To interpret in **original units** (e.g., "extra dollars spent per extra dollar of income"), convert back using `sigma`.


In [ ]:
print(f"{'Feature':30s}{'w (scaled)':>14s}{'w (per original unit)':>24s}")
for name, w_s, sig in zip(feature_names, w_final, sigma):
    w_original_units = w_s / sig
    print(f"{name:30s}{w_s:14.4f}{w_original_units:24.5f}")

# Recover an intercept usable directly with RAW (unscaled) features, for convenience:
b_original = b_final - np.sum((w_final / sigma) * mu)
w_original = w_final / sigma
print(f"\nEquivalent model on raw units:  y_hat = {b_original:.3f} + " +
      " + ".join(f"{w:.5f}*{name}" for w, name in zip(w_original, feature_names)))

<a name="toc_10"></a>
# 10 Predict on New Data

Edit `new_observation` below to match your own use case (must have the same features, in the same order, as `feature_names`).


In [ ]:
new_observation = np.array([3000, 4, 13, 42])  # EDIT THIS for your project

new_scaled, _, _ = zscore_normalize(new_observation.reshape(1, -1), mu, sigma)
prediction = predict(new_scaled, w_final, b_final)[0]

print(f"New observation: {dict(zip(feature_names, new_observation))}")
print(f"Predicted {target_name}: {prediction:.2f}")

<a name="toc_11"></a>
# 11 Diagnostics: Residual Plots

A quick visual check for problems: residuals should look like unstructured noise scattered around zero. A clear pattern (curve, funnel shape, trend) suggests a missing nonlinear term, heteroskedasticity, or an omitted variable — common issues in applied economic regressions.


In [ ]:
residuals = y_train - y_train_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].scatter(y_train_pred, residuals)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xlabel('Predicted value')
axes[0].set_ylabel('Residual (actual - predicted)')
axes[0].set_title('Residuals vs. Predicted')

axes[1].hist(residuals, bins=min(10, len(residuals)))
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual distribution')
plt.show()

---
## Checklist before you trust this model for a real economics project

- [ ] Enough observations relative to features ($m \gg n$ — this template's toy example violates this on purpose; real work needs much more data)
- [ ] Features scaled, and `mu`/`sigma` from **training data only**
- [ ] Cost curve has flattened (converged), not still falling or diverging
- [ ] Reasonable test-set performance, not just training performance
- [ ] Residuals look like noise, no obvious pattern
- [ ] Coefficients have plausible economic signs/magnitudes
- [ ] You're not claiming causation from correlation without a research design that supports it (randomized/natural experiment, instrumental variables, panel fixed effects, etc.)
